# CycleGAN Ergebnis-Viewer (bestehende Outputs)

Wähle ein Experiment unter `results/cyclegan_inference_demo/<experiment>` und zeige die bereits generierten Bilder (day2night/night2day) zusammen mit den Originalen. Es wird **nichts neu berechnet** und **nichts gespeichert**.


In [1]:
from pathlib import Path
import random
import matplotlib.pyplot as plt
from PIL import Image

# --- Pfad-Setup ---
def find_repo_root():
    markers = ['.git', 'src', 'results', 'configs']
    start = Path.cwd()
    for path in [start, *start.parents]:
        if any((path / m).exists() for m in markers):
            return path.resolve()
    return start.resolve()

repo_root = find_repo_root()
print('Arbeitsverzeichnis:', Path.cwd())
print('Repo-Root:', repo_root)

# --- Parameter ---
#experiment_name = 'results/cyclegan_inference_demo/cyclegan_day2night_20251207_053414'  # Experiment-Ordner unter results/cyclegan_inference_demo
experiment_name = 'results/cyclegan_inference_demo/cyclegan_day2night_20251211_015934' # strasenbilder unf fh nacht
#experiment_name = 'results/cyclegan_inference_demo/cyclegan_day2night_20251208_020426'
results_root = repo_root / experiment_name
directions = ['day2night', 'night2day']
samples_per_direction = None  # None = alle Ausgaben zeigresults/cyclegan_inference_demo/cyclegan_day2night_20251208_020426en; Zahl = zufällige Auswahl
originals = {
    'day2night': repo_root / 'data/bdd_split/cyclegan/testA',
    'night2day': repo_root / 'data/bdd_split/cyclegan/testB',
}
print('Experiment:', experiment_name)
print('Results root:', results_root)


Arbeitsverzeichnis: /srv/store/docker-users/thesis/khajuria/day2night/results/cyclegan_inference_demo
Repo-Root: /srv/store/docker-users/thesis/khajuria/day2night
Experiment: results/cyclegan_inference_demo/cyclegan_day2night_20251211_015934
Results root: /srv/store/docker-users/thesis/khajuria/day2night/results/cyclegan_inference_demo/cyclegan_day2night_20251211_015934


In [2]:
def direction_root(direction: str) -> Path:
    return results_root / direction


def gather_pairs(direction: str, count: int | None):
    out_dir = direction_root(direction)
    if not out_dir.is_dir():
        print(f'Kein Ordner für {direction}: {out_dir}')
        return []
    orig_root = originals.get(direction)
    if not orig_root or not orig_root.is_dir():
        print(f'Kein Original-Ordner für {direction}: {orig_root}')
        return []
    imgs = [p for p in sorted(out_dir.rglob('*')) if p.is_file() and p.suffix.lower() in {'.jpg','.jpeg','.png','.bmp','.tif','.tiff'}]
    if not imgs:
        print(f'Keine Bilder in {out_dir}')
        return []
    if count is not None:
        random.shuffle(imgs)
        imgs = imgs[:count]
    pairs = []
    for path in imgs:
        rel = path.relative_to(out_dir)
        orig = orig_root / rel
        if not orig.is_file():
            continue
        try:
            out_img = Image.open(path).convert('RGB')
            orig_img = Image.open(orig).convert('RGB')
        except Exception as e:
            print(f'Überspringe {path}: {e}')
            continue
        pairs.append((path, orig_img, out_img, orig))
    if not pairs:
        print(f'Keine passenden Originale für {direction}')
    return pairs


def show_pairs(pairs, title: str):
    if not pairs:
        return
    n = len(pairs)
    fig, axes = plt.subplots(n, 2, figsize=(10, 4 * n))
    if n == 1:
        axes = [axes]
    for ax_row, (path, orig, out_img, orig_path) in zip(axes, pairs):
        ax_row[0].imshow(orig)
        ax_row[0].set_title(f'Original\n{orig_path.name}')
        ax_row[0].axis('off')
        ax_row[1].imshow(out_img)
        ax_row[1].set_title(f'{title}\n{path.name}')
        ax_row[1].axis('off')
    fig.tight_layout()
    plt.show()


In [3]:
for direction in directions:
    count = None if samples_per_direction is None else int(samples_per_direction)
    pairs = gather_pairs(direction, count)
    show_pairs(pairs, direction.replace('2', ' → '))




Keine passenden Originale für day2night
Kein Ordner für night2day: /srv/store/docker-users/thesis/khajuria/day2night/results/cyclegan_inference_demo/cyclegan_day2night_20251211_015934/night2day


In [4]:
# Streckt auf exakten 16:9-Frame (kein Padding)
def stretch_exact_169(img, aspect=16 / 9):
    w, h = img.size
    # try keeping height, stretch width to 16:9
    target_w = round(h * aspect)
    target_h = h
    # if that would shrink width, instead keep width and stretch height
    if target_w < w:
        target_w = w
        target_h = round(target_w / aspect)
    return img.resize((target_w, target_h), Image.BICUBIC)


def show_pairs_stretched_169(pairs, title: str):
    if not pairs:
        return
    stretched = [
        (path, orig_img, stretch_exact_169(out_img), orig_path)
        for (path, orig_img, out_img, orig_path) in pairs
    ]
    show_pairs(stretched, f"{title} (gestreckt 16:9)")

for direction in directions:
    count = None if samples_per_direction is None else int(samples_per_direction)
    pairs = gather_pairs(direction, count)
    show_pairs(pairs, direction.replace('2', ' → '))

    if direction == 'day2night':
        show_pairs_stretched_169(pairs, 'day → night')


Keine passenden Originale für day2night
Kein Ordner für night2day: /srv/store/docker-users/thesis/khajuria/day2night/results/cyclegan_inference_demo/cyclegan_day2night_20251211_015934/night2day
